In [81]:
import networkx as nx
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import re
import requests
import os
from datetime import datetime
import json

# 1. API Testing for additional data

API rate limits take into account client identity to determine the level of access.
Stronger forms of identification result in a higher limit, such as running in Wikimedia Cloud Services (WMCS) or authenticating requests.
The highest limits require running in WMCS, community bot approval, or being well-known to the Wikimedia Foundation.
Further limits or additional best practices may apply to specific APIs.

## 1.1 Create dataset

In [ ]:
def create_dataset(input_csv, output_csv):

    csv_exists = os.path.exists(input_csv)

    if not csv_exists:
        raise FileNotFoundError(f"{input_csv} not found")

    print(f"✓ Found input CSV: {input_csv}")

    # Load existing data
    admins_df = pd.read_csv(input_csv)

    admins = admins_df["Admin"].drop_duplicates().tolist()

    print(f"✓ Loaded {len(admins)} administrators")

    # update dataframe
    admins_df = pd.DataFrame({
        "Admin": pd.Series(admins, dtype="string"),
        "Active": pd.Series(pd.NA, dtype="Int64"),
        "Semi-Active": pd.Series(pd.NA, dtype="Int64"),
        "Inactive": pd.Series(pd.NA, dtype="Int64"),
        "editcount": pd.Series(pd.NA, dtype="Int64"),
        "registration_date": pd.Series(pd.NA, dtype="string"),
        "groups": pd.Series(pd.NA, dtype="string"),
        "blocked": pd.Series(pd.NA, dtype="boolean"),
        "gender": pd.Series(pd.NA, dtype="string")
    })

    # Always overwrite
    admins_df.to_csv(output_csv, index=False)
    print(f"✓ Updated {output_csv}")
    return admins_df

# example usage
# admins_df = create_dataset("rfa_graph.csv", "rfa_graph.csv")

✓ Found input CSV: rfa_graph.csv
✓ Loaded 10416 administrators
✓ Updated rfa_graph.csv


## 1.1.1 Delete columns

If we want to delete a columns of a dataset, we can use this function

In [ ]:
def delete_column(df, column_name):
    if column_name in df.columns:
        df.drop(columns=[column_name], inplace=True)
        print(f"✓ Deleted column: {column_name}")
    else:
        print(f"⚠️ Column '{column_name}' not found in DataFrame")

## 1.2 Data Collection for each admin

Now we need to fetch structured metrics for multiple admins. We attempt to fetch the following metrics:

- ```Editcount```: Number of edits the user has made since their ```Registration```.
- ```Groups```: The names of groups the user is a member of (can't differentiate between active and inactive).
- ```Registration```: The date of the user's registration, formatted as YYYY-MM-DD.
- ```Rights```: The abilities a user has on Wikipedia.
- ```Blockinfo```: Names of other users the user has blocked due to reasons.

In [ ]:
def fetch_wikipedia_admin_metrics(input_csv, output_csv, batch_size=50, test_mode=False):

    """
    Fetch edit count and registration date for Wikipedia admins using the MediaWiki API.

    Parameters
    ----------
    input_csv : str
        Path to CSV containing an 'Admin' column.
    output_csv : str or None
        Path to save updated CSV.
        If None, overwrites input_csv.
    batch_size : int
        Number of usernames per API request (max 50).

    Returns
    -------
    pd.DataFrame
        Updated dataframe.
    """

    if output_csv is None:
        output_csv = input_csv

    # Load admins
    admins_df = pd.read_csv(input_csv)
    admins_df["groups"] = admins_df["groups"].astype("string")
    admins_df["gender"] = admins_df["gender"].astype("string")
    admins_df["registration_date"] = admins_df["registration_date"].astype("string")
    admins_df["editcount"] = pd.to_numeric(admins_df["editcount"], errors="coerce").astype("Int64")
    admins_df["blocked"] = admins_df["blocked"].astype("boolean")
    admins = admins_df["Admin"].tolist()

    headers = {
        "User-Agent": "CSS_ProjectBot/1.0 (https://github.com/Stampe04/CSS_Project; fwaggot.player@gmail.com)",
        "From": "fwaggot.player@gmail.com",
        "Accept": "application/json"
    }

    url = "https://en.wikipedia.org/w/api.php"

    all_users = []

    total = len(admins)
    next_print_10pct = 0.1

    print(f"Dataset: {input_csv}\nFetching metrics for {total} administrators...")

    for i in range(0, total, batch_size):

        if test_mode and i >= 100:
            print("Test mode: stopping after 100 admins")
            break

        batch = admins[i:i + batch_size]

        params = {
            "action": "query",
            "list": "users",
            "ususers": "|".join(batch),
            "usprop": "editcount|registration|groups|blockinfo|rights|gender",
            "format": "json"
        }

        try:
            response = requests.post(url, headers=headers, data=params, timeout=10)
            response.raise_for_status()

            users = response.json()["query"]["users"]
            all_users.extend(users)

            # ---- progress tracking ----
            processed = min(i + batch_size, total)

            # print every 10%
            if processed / total >= next_print_10pct:
                print(f"✓ Progress: {int(processed/total*100)}%")
                next_print_10pct += 0.1

        except Exception as e:
            print(f"Error fetching batch {i // batch_size + 1}: {e}")

    # Update dataframe
    for user in all_users:

        username = user.get("name")

        mask = admins_df["Admin"] == username

        if not mask.any():
            continue

        groups = user.get("groups") or []
        admins_df.loc[mask, "groups"] = "|".join(groups)
        admins_df.loc[mask, "blocked"] = bool(user.get("blockinfo"))
        admins_df.loc[mask, "gender"] = user.get("gender") or pd.NA
        admins_df.loc[mask, "editcount"] = user.get("editcount")
        admins_df.loc[mask, "registration_date"] = user.get("registration") or pd.NA

    # Save updated dataframe
    admins_df.to_csv(output_csv, index=False)
    print(f"✓ Updated {output_csv}")

    return admins_df

# example usage
# admins_df = fetch_wikipedia_admin_metrics("rfa_graph.csv", "rfa_graph.csv", batch_size=50, test_mode=True)

Dataset: rfa_graph.csv
Fetching metrics for 10416 administrators...
Test mode: stopping after 100 admins
✓ Updated rfa_graph.csv


## 1.3 Update the Wikipedia_Admins.csv file

Cross-reference and extract admin names from the communities for each category:

- ```Active```, 
- ```Semi-Active``` and 
- ```Inactive```.

Then update the admin list to further develop the community graphs. This should give us a better understanding of the communites *(not temporal as the different states only detects from the past 2 months)*.

In [ ]:
def extract_and_categorize_admins(input_csv, output_csv, test_mode=False):

    headers = {
        "User-Agent": "CSS_ProjectBot/1.0 (https://github.com/Stampe04/CSS_Project; fwaggot.player@gmail.com)",
        "From": "fwaggot.player@gmail.com",
        "Accept": "application/json"
    }

    url = "https://en.wikipedia.org/w/api.php"

    activity_categories = {
        "Active": set(),
        "Semi-Active": set(),
        "Inactive": set()
    }

    category_pages = {
        "Active": "Active",
        "Semi-Active": "Semi-active",
        "Inactive": "Inactive"
    }

    # Load admins
    admins_df = pd.read_csv(input_csv)
    admins_df["groups"] = admins_df["groups"].astype("string")
    admins_df["gender"] = admins_df["gender"].astype("string")
    admins_df["registration_date"] = admins_df["registration_date"].astype("string")
    admins_df["editcount"] = pd.to_numeric(admins_df["editcount"], errors="coerce").astype("Int64")
    admins_df["blocked"] = admins_df["blocked"].astype("boolean")
    admins_df = admins_df["Admin"].tolist()

    print(f"Dataset: {input_csv}\nExtracting admin activity categories...")

    for category, suffix in category_pages.items():

        params = {
            "action": "parse",
            "page": f"Wikipedia:List_of_administrators/{suffix}",
            "format": "json"
        }

        try:
            response = requests.post(
                url,
                headers=headers,
                params=params,
                timeout=10
            )

            response.raise_for_status()

            html = response.json()["parse"]["text"]["*"]

            usernames = re.findall(
                r'href="/wiki/User(?:_talk)?:([^"]+)"',
                html
            )

            usernames = {
                name.replace("_", " ").strip()
                for name in usernames
            }

            activity_categories[category] = usernames

            print( 
                f"Before time {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} - "
                f"{category}: "
                f"{len(usernames)} admins"
            )

        except Exception as e:

            print(
                f"Error fetching {category}: {e}"
            )

    # Update dataframe
    for category in activity_categories:

        if category not in admins_df.columns:
            admins_df[category] = 0

        admins_df[category] = (
            admins_df["Admin"]
            .isin(activity_categories[category])
            .astype(int)
        )

    admins_df.to_csv(
        output_csv,
        index=False
    )

    return admins_df

# example usage
# admins_df = extract_and_categorize_admins("rfa_graph.csv", "rfa_graph.csv", test_mode=False)

Dataset: rfa_graph.csv
Extracting admin activity categories...
Before time 2026-05-06 01:54:05 - Active: 416 admins
Before time 2026-05-06 01:54:06 - Semi-Active: 317 admins
Before time 2026-05-06 01:54:06 - Inactive: 76 admins


## 1.4 Create the full dataset

Now we combine the two functions and create the whole dataset we can use for communities. While we still haven't optimized the code and can tidy up the design, it's a marvel that we haven't crashed or used to many units to create this. Now we simply run four cells and can create an entire dataset.

In [104]:
def build_wikipedia_admin_dataset(dataset_csv):

    admins_df = fetch_wikipedia_admin_metrics(
        test_mode=False, 
        input_csv=dataset_csv, 
        output_csv=dataset_csv
    )

    admins_df = extract_and_categorize_admins(
        input_csv=dataset_csv,
        output_csv=dataset_csv
    )

    return admins_df

admins_df =build_wikipedia_admin_dataset("wikipedia_admins.csv")

Dataset: wikipedia_admins.csv
Fetching metrics for 10416 administrators...
✓ Progress: 10%
✓ Progress: 20%
✓ Progress: 30%
✓ Progress: 40%
✓ Progress: 50%
✓ Progress: 60%
✓ Progress: 70%
✓ Progress: 80%
✓ Progress: 90%
✓ Progress: 100%
✓ Updated wikipedia_admins.csv
Dataset: wikipedia_admins.csv
Extracting admin activity categories...
Before time 2026-05-06 02:01:07 - Active: 416 admins
Before time 2026-05-06 02:01:08 - Semi-Active: 317 admins
Before time 2026-05-06 02:01:08 - Inactive: 76 admins


# 5.1 Make functions convertable/universal for .json files

Now that we can extract all the relevant information on the administrators given our MediaWiki API, we need to make sure it also works on our given ```rfa_graph.json``` file (although we might need to tweak and/or clean the file).

In [ ]:
# Load RFA graph from JSON file
with open("rfa_graph.json", "r") as f:
    try:
        rfa_graph = json.load(f)
        print("✓ RFA Graph loaded successfully")
    except json.JSONDecodeError as e:
        print(f"Error loading RFA Graph: {e}")

# Extract admin-target relationships
rows = [
    {"Admin": edge["source"], "Target": edge["target"]}
    for edge in rfa_graph["edges"]
]

# Create dataframe
rfa_df = pd.DataFrame(rows)

# Group targets by admin
rfa_df = (
    rfa_df
    .groupby("Admin")["Target"]
    .apply(lambda x: x.drop_duplicates().tolist())
    .reset_index()
)

# Save
rfa_df.to_csv("rfa_graph.csv", index=False)

print(f"✓ Extracted {len(rfa_df)} unique admins")
print(rfa_df.head())

✓ RFA Graph loaded successfully
✓ Extracted 10416 unique admins
               Admin                                             Target
0  !---slappdash---!                                           [Tedder]
1             %D0%90  [Ambush Commander, Bbatsell, Can't sleep, clow...
2               'sed               [Yanksox, Crisspy, SynergeticMaggot]
3              (.Y.)                                       [TomTheHand]
4         (:Julien:)                                      [Sam Hocevar]
